Notebook para generar embeddings
Previamente ya se realizaron para pocos archivos usando un modelos de OpenAI y validando con Qdrant [rag_qdrant](https://github.com/Halsey26/embedding_PerAI/blob/main/rag_qdrant.ipynb)
Sin embargo, ahora son más de 30 archivos pdf, algunos incluso con 300 páginas. Por ende se plantea usar langchain para:
- Chunkenizado
- Embedding
- Almacenamiento - Qdrant
- Función búsqueda
Después se modularizará para detectar los pdfs y obtener los embeddings

Librerias para descargar
- %pip install -qU pypdf
- pip install langchain
- pip install langchain-community
- pip install sentence-transformers


## fsdf
Detecta si un pdf ya ha sido procesado. Si en caso no ha sido procesado, se aplica las funciones y se marca como **hecho**.

In [81]:
# se crea un archivo .txt para almacenar los nombres de los archivos ya procesados
import os

if not os.path.exists('procesados.txt'):
    with open('procesados.txt', 'w') as file:
        pass # crea un archivo vacio

In [11]:
# lee los archivos procesados, por defecto nada
with open('procesados.txt', 'r') as file:
    procesados= set(file.read().splitlines())

procesados

{''}

In [5]:
procesados_2= {}

In [26]:
import os
import hashlib

ruta_docs_pdf= '../doc_pdf'
ruta_docs_procesados= '../docs_procesados'
# carpeta_embeddings = ''

docs_no_procesados= []
# verificamos los archivos en carpeta de docs
for filename in os.listdir(ruta_docs_pdf):
    # verificar si el archivo se encuentra en procesados.txt
    if filename not in procesados:
        # print('El archivo no ha sido procesado')
        ruta_completa= os.path.join(ruta_docs_pdf,filename)
        docs_no_procesados.append(ruta_completa)
    else:
        print('Todos los archivos han sido procesados')

print(f'Documentos para procesar:\n  {docs_no_procesados}')
# luego que finalice todo el proceso, hay que realizar una función para agregar el archivo a procesados.txt
# with open('procesados.txt', 'w') as file:
#                 file.write(filename+"\n")

# ruta= os.path.join(ruta_docs_pdf, filename)

# doc_id2 = hashlib.md5(ruta.encode()).hexdigest() # codificamos la entrada string, aplicamos algoritmo y obtenemos salida hexadecimal


Documentos para procesar:
  ['../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf', '../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']


In [74]:
# generamos una función, para los id de cada documento pdf
import hashlib

ruta= os.path.join(ruta_docs, filename)

doc_id = hashlib.md5(ruta.encode()).hexdigest() # codificamos la entrada string, aplicamos algoritmo y obtenemos salida hexadecimal
doc_id



'126cf3d9d317cfba54f592249e254a09'

## Empieza el procesamiento

Extracción del texto 

In [ ]:
import time
from langchain_community.document_loaders import PyPDFLoader

def extraccion_page(ruta):
    loader = PyPDFLoader(ruta)
    pages = loader.load()
    # async for page in loader.alazy_load():
    #     pages.append(page)
    return pages


/usr/local/python/3.12.1/lib/python3.12/site-packages/pypdf/_utils.py:265: RuntimeWarning: coroutine 'extraccion_page' was never awaited
  m = regex.search(name + tok)
/usr/local/python/3.12.1/lib/python3.12/site-packages/pypdf/_utils.py:265: RuntimeWarning: coroutine 'extraccion_page_async' was never awaited
  m = regex.search(name + tok)


Limpieza del texto

In [43]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve símbolos de copyright y similares
    text = re.sub(r'\n+', ' ', text)  # convierte múltiples saltos de línea en espacio
    text = re.sub(r'\s{2,}', ' ', text)  # remueve espacios extra
    return text.strip()


Creacción de la metadata, estructura planteada:
- documento_id
- nombre documento
- numero pagina
- total_pages

In [55]:
from pathlib import Path

def generate_metadata(ruta_completa, pages):
    filename = Path(ruta_completa).name
    document_id = hashlib.md5(filename.encode()).hexdigest() # codificamos solo el nombre del archivo
    total_pages = pages[0].metadata['total_pages']
    docs_metadata = []
    for page in pages:
        page_number = page.metadata['page_label'] # númeración correcta de la página
        
        metadata = {
            "document_id": document_id,
            "filename": filename,
            "page_number": page_number,
            "total_pages": total_pages,
        }
        page.page_content = clean_text(page.page_content) # cleaned_text = clean_text(page.page_content)
        
        docs_metadata.append(
            {
                'text': page.page_content, #cleaned_text, 
                'metadata': metadata
            }
        )
    return docs_metadata

Generación de embeddings

Función completa 

In [ ]:
prueba = ['../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf']

for ruta_archivo in  prueba:#docs_no_procesados:
    # definir una funcion para aplicar  
    time1= time.time()
    pags=extraccion_page(ruta_archivo)
    docs_metadata = generate_metadata(ruta_archivo, pags)
    time3=time.time()
    print(f'Tiempo total por {ruta_archivo}: {time3-time1:.2f} segundos')
    print('✅ Realizado: extracción, limpieza del texto y generación de metadata\n')

# Funcion completa
# def procesamiento():
    # extracion texto
    # limpieza por pagina
    # creacion de docs_metadata
    # obtencion de embedding
    

Tiempo total por ../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf: 14.48 segundos
✅ Realizado: extracción, limpiezadel texto y generación de metadata



In [60]:
docs_metadata

[{'text': 'Machine Translated by Google',
  'metadata': {'document_id': 'bc9b46fc1eb0b8290429faeaedef6aed',
   'filename': '601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf',
   'page_number': '1',
   'total_pages': 386}},
 {'text': 'Machine Translated by Google',
  'metadata': {'document_id': 'bc9b46fc1eb0b8290429faeaedef6aed',
   'filename': '601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf',
   'page_number': '2',
   'total_pages': 386}},
 {'text': 'A mis fantásticos padres y mi increíble hermana, gracias por todo el apoyo y confianza a lo largo de los años. PD Para cualquier hijo que nazca en el futuro, lamento que no estés en esta dedicatoria. A mi encantadora esposa Jennifer y mi hijo Liav. De repente, todo vuelve a ser posible. Estarías aquí si ya existieras. Machine Translated by Google',
  'metadata': {'document_id': 'bc9b46fc1eb0b8290429faeaedef6aed',
   'filename': '601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf',
   'page_number': '3',
   'total_pages': 

In [36]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
api_key=os.getenv('OPENAI_API_KEY')
# api_key
cliente= OpenAI()
cliente

In [62]:
from sentence_transformers import SentenceTransformer

docs_embedd = []
modelo_seleccionado= SentenceTransformer('BAAI/bge-large-en-v1.5')

modelo_openai = "text-embedding-3-small"

for doc in docs_metadata:
    '''
     cuando tenga el modelo habilitado:
    response = cliente.embeddings.create(
        input= doc['text'], 
        model = modelo_openai
    )

    embedding= response.data[0].embedding
    '''
    embedding= modelo_seleccionado.encode(doc['text'], normalize_embeddings= True)
    docs_embedd.append({
        'vector': embedding.tolist(),  #con openai, directamente el embedding
        'text': doc['text'], 
        'metadata': doc['metadata']
    })

/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Ya ahora que tengo el embedding demo vamos a modularizar

In [66]:
docs_embedd

[{'vector': [0.0497138611972332,
   0.04694351926445961,
   -0.044723354279994965,
   -0.022731542587280273,
   -0.06637521833181381,
   -0.02240157686173916,
   0.017499852925539017,
   -0.015371651388704777,
   0.02037905342876911,
   0.04144832491874695,
   -0.01189467590302229,
   0.012864307500422001,
   0.04108339920639992,
   -0.002206462202593684,
   -0.01731467805802822,
   0.01215458009392023,
   -0.052837684750556946,
   -0.02681773528456688,
   -0.03021322563290596,
   0.012331689707934856,
   0.011663584969937801,
   -0.021653378382325172,
   -0.06614203006029129,
   -0.037349723279476166,
   0.002462279750034213,
   0.05117793753743172,
   0.00537277664989233,
   0.021219402551651,
   0.0701344758272171,
   0.02696160040795803,
   -0.019501227885484695,
   0.017046667635440826,
   0.057447779923677444,
   -0.00012350759061519057,
   -0.03930116444826126,
   0.011558870784938335,
   -0.008937196806073189,
   -0.04051511734724045,
   -0.006343390792608261,
   -0.01342871598

In [67]:
embedding[1000]

np.float32(0.0060975687)

In [43]:
from sentence_transformers import SentenceTransformer
import time

modelo_seleccionado= SentenceTransformer('BAAI/bge-large-en-v1.5')
text = 'The EMyth Business Development Path 1 The EMyth Business Development Path'
start= time.time()
embedding= modelo_seleccionado.encode(text, normalize_embeddings= True)
finish= time.time()

embedding

# text = "passage: Your text to embed"
# embedding = model.encode(text, normalize_embeddings=True)


/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


array([ 0.04971386,  0.04694352, -0.04472335, ..., -0.02569201,
       -0.00941801,  0.01352354], shape=(1024,), dtype=float32)

In [68]:
print(f"Tiempo: {finish - start:.2f} segundos")
len(embedding)

Tiempo: 7.12 segundos


1024

In [3]:
import langchain
import langchain_community